# X6Y3 experiments in LeeQ

Use the supplied, already largely calibrated notebook parameters. This notebook preserves independent X/X90 FAST_DRAG pulses, virtual-Z corrections and calibrated readout windows. It does not retune or write back calibration. Q0/Q1 only, on the existing Huracan aa01c78f service. The original qcal notebook targets eight qubits; this subset is unheralded, uses passive preparation, and omits EF/CZ/active reset.

Run All is offline by default. `RUN_HARDWARE = True` enables real shots through the existing RPC service; first confirm the instrument is available and loaded image/channel metadata match. No firmware/software deployment, service restart or RF configuration change is part of this notebook.


In [ ]:
import os
from pathlib import Path
import sys
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'

# Adapt paths on another development host. No board-side installation is needed.
LEEQ_ROOT = Path('/local/data/projects/LeeQ-lbnl-qubic')
HARNESS_ROOT = Path('/local/data/projects/qubic_validation')
CLIENT_PACKAGES = HARNESS_ROOT / 'artifacts/x6y3/venv/lib/python3.11/site-packages'
sys.path.insert(0, str(LEEQ_ROOT))
sys.path.append(str(CLIENT_PACKAGES))
CALIBRATION = HARNESS_ROOT / 'X6Y3/configs_new/config.yaml'
CHANNEL_CONFIG = Path('/local/data/artifacts/zcu216/20260903T_aa01c78f_rdrv230_d4_candidate/bits_aa01c78f/channel_config.json')
RUN_HARDWARE = False
QUBIT = 0


In [ ]:
import json
import numpy as np
from datetime import datetime, timezone
from leeq.setups.huracan import create_huracan_setup
from leeq.setups.x6y3 import X6Y3Calibration
from leeq.experiments.experiments import ExperimentManager
from leeq.experiments.x6y3 import (
    readout_plan, amplitude_plan, ramsey_plan, t1_plan,
    acquire_plan, analyze_readout, analyze_sweep,
)
from leeq.experiments.x6y3_validation import verify_plan_parity

calibration = X6Y3Calibration(CALIBRATION)
huracan = create_huracan_setup(CHANNEL_CONFIG)
calibration.verify_channel_metadata(huracan.channel_metadata)
print('Calibration SHA256:', calibration.sha256)
print('Source heralding:', calibration.source_herald, '| LeeQ subset: unheralded')
print('Passive wait (us):', calibration.passive_delay_us)


## Offline checks

These checks use native LeeQ LPBs and an independent qcal transpilation reference. They compare scheduled pulses and assembled waveform/frequency memories. No acquisition is performed. Complete grids for both qubits can also be checked with `scripts/check_x6y3_reproduction.py --full`.


In [ ]:
checks = [
    readout_plan(QUBIT, shots=17),
    amplitude_plan(QUBIT, scales=[0.7, 1.0, 1.3]),
    ramsey_plan(QUBIT, delays_us=[0, 0.123, 1], detunings_mhz=[-2.5, 2.5]),
    t1_plan(QUBIT, delays_us=[0, 17.25, 350]),
]
for plan in checks:
    report = verify_plan_parity(calibration, huracan, plan)
    print(plan.name, report['points_verified'], 'points matched')


## Baseline readout

Ground and nominal excited (two X90 pulses) preparations alternate by block. The classifier is fit on earlier blocks and tested on later blocks. Its score is assignment to nominal preparation labels, not independent qubit-state fidelity. Existing qcal pickles are preserved and are not loaded by this notebook.


In [ ]:
results = {}
if RUN_HARDWARE:
    import socket
    socket.setdefaulttimeout(60)
    manager = ExperimentManager()
    if manager.get_default_setup() is not huracan:
        manager.register_setup(huracan)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
    output = HARNESS_ROOT / 'artifacts/x6y3/leeq-runs' / stamp
    pilot = acquire_plan(calibration, huracan, readout_plan(QUBIT, shots=17), output / 'pilot')
    print('Pilot IQ shape:', pilot['iq'].shape)
    results['readout'] = acquire_plan(calibration, huracan, readout_plan(QUBIT, shots=512, blocks=4), output / 'readout')
    readout_report = analyze_readout(results['readout'])
    (output / 'readout/analysis.json').write_text(json.dumps(readout_report, indent=2))
    print(readout_report)
else:
    print('Offline mode: no shots submitted.')


In [ ]:
if RUN_HARDWARE:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 5))
    for state, label in [(0, 'ground preparation'), (1, 'X90 + X90 preparation')]:
        iq = results['readout']['iq'][state::2].ravel()
        ax.scatter(iq.real, iq.imag, s=3, alpha=0.25, label=label)
    ax.set(xlabel='I (raw units)', ylabel='Q (raw units)', title=f'Q{QUBIT} baseline readout')
    ax.legend()
    plt.show()


## Baseline-centered diagnostics

These cells characterize the supplied calibration. Amplitude uses a fixed-duration four-X90 sequence with a 0.7–1.3 scale sweep. Ramsey uses the original virtual-Z detuning protocol. T1 uses two X90 pulses and 0–350 us delays. Fits report observations or candidate corrections only; nothing writes them into calibration. Inspect readout quality before running these cells.


In [ ]:
if RUN_HARDWARE:
    plans = [amplitude_plan(QUBIT), ramsey_plan(QUBIT), t1_plan(QUBIT)]
    for plan in plans:
        result = acquire_plan(calibration, huracan, plan, output / plan.name)
        analysis = analyze_sweep(result, readout_report)
        (output / plan.name / 'analysis.json').write_text(json.dumps(analysis, indent=2))
        results[plan.name] = result
        print(plan.name, analysis.get('fits', []), analysis.get('fit_error', ''))


In [ ]:
if RUN_HARDWARE:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, name in zip(axes, ['amplitude', 'ramsey', 't1']):
        result = results[name]
        analysis = analyze_sweep(result, readout_report)
        points = result['manifest']['points']
        xkey = 'amplitude_scale' if name == 'amplitude' else 'delay_us'
        x = np.array([p[xkey] for p in points])
        y = np.array(analysis['mean'])
        error = np.array(analysis['standard_error'])
        if name == 'ramsey':
            for detuning in sorted({p['detuning_mhz'] for p in points}):
                mask = np.array([p['detuning_mhz'] == detuning for p in points])
                ax.errorbar(x[mask], y[mask], error[mask], fmt='.-', label=f'{detuning} MHz')
            ax.legend()
        else:
            ax.errorbar(x, y, error, fmt='.-')
        ax.set(title=name, xlabel=xkey, ylabel='Baseline IQ projection')
    fig.tight_layout()
    fig.savefig(output / 'diagnostics.png', dpi=160)
    plt.show()
